# A2A Protocol Foundations: Agent Cards, Tasks, and Messages

**A2A (Agent-to-Agent protocol)** is an open, vendor-neutral protocol (originated at Google, now under the Linux Foundation) that lets independently built agents talk to each other over HTTP, regardless of which framework or vendor built them. Where MCP (`09_Agent_Protocols/MCP/`) standardizes how a *single* agent talks to *tools and data*, A2A standardizes how *one agent talks to another agent* -- discovery, capability negotiation, and task hand-off.

This notebook is **Part 1 of 3** in this repo's A2A track:

| Notebook | Scope |
|---|---|
| `01_Foundations/01_A2A_Protocol_Basics.ipynb` (this notebook) | Protocol mechanics: Agent Cards, Tasks, Messages |
| `02_Building_Agents_with_A2A/` | An actual LLM-backed agent exposed over A2A |
| `03_Applications/` | Cross-framework agent interop demo |

The goal here is to understand the **protocol**, not to build a smart agent -- the "agent logic" we expose below is a deliberately trivial echo responder. Everything else (the server scaffolding, the Agent Card, the client discovery/call flow) is the real, official `a2a-sdk` package, the same one already used elsewhere in this repo at `09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/utilities/a2a/`.

## What We Are Going to Do

1. Explain the three core A2A building blocks: **Agent Card**, **Task**, **Message**.
2. Stand up a minimal, real A2A **server** using `a2a-sdk` -- it serves an `AgentCard` at `/.well-known/agent.json` and answers messages with a canned echo reply.
3. Run that server **in-process** (as an `asyncio` background task inside this notebook's own event loop) so the same notebook can also act as the client.
4. Build a minimal A2A **client**: use `A2ACardResolver` to *discover* the agent's capabilities, then use `A2AClient` to *submit a task* (a message) and read back the response.
5. Narrate, at each step, what is happening in protocol terms: discovery -> capability negotiation -> task submission -> response.

## Core A2A Concepts

### 1. Agent Card
A JSON manifest that an agent publishes at a well-known path -- by convention `/.well-known/agent.json`. It is the agent's business card: name, description, the URL to call, the transport/protocol version it speaks, its declared `capabilities` (e.g. streaming, push notifications), and its `skills` (named abilities, each with a description, example prompts, and input/output MIME types). A client fetches this card **before** it ever sends a real request -- this is how two agents that have never met agree on how to talk to each other ("capability negotiation").

### 2. Task
The unit of work sent to an agent. A task has a server-generated `id`, a `context_id` (groups related tasks/turns together), and a lifecycle expressed as a `TaskState`: `submitted -> working -> (input-required) -> completed | failed | canceled`. Not every exchange needs a full task -- a single request/response turn can also resolve as a plain `Message` instead of a `Task` object, which is what our trivial echo agent will do.

### 3. Message
The conversational turn exchanged within a task. A `Message` has a `role` (`user` or `agent`) and a list of `Part`s (a `TextPart` for plain text, but also `FilePart`/`DataPart` exist in the spec for richer payloads). Sending a task, under the hood, means sending a `message/send` JSON-RPC request whose params carry one `Message`.

> **Note on production use:** the agent below returns a hardcoded string so the protocol mechanics stay front and center. A real A2A-compliant agent would instead run its message text through an LLM call inside the same handler -- in this repo, that means `from helpers import get_llm` and invoking `get_llm()` on `context.get_user_input()` instead of echoing it. That wiring is exactly what `02_Building_Agents_with_A2A/` (sibling notebook, authored separately) will demonstrate.

## Environment Setup

This notebook needs the official `a2a-sdk` package plus an ASGI server (`uvicorn`) to actually serve HTTP, and `httpx` for the client's async HTTP calls. `a2a-sdk>=0.3.0` is already a pinned dependency of `09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/` elsewhere in this repo.

In [ ]:
%pip install -q "a2a-sdk>=0.3.0" uvicorn httpx

In [ ]:
# ============ IMPORTS ============
import asyncio
import uuid

import httpx
import uvicorn

from a2a.client import A2ACardResolver, A2AClient
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
    Message,
    MessageSendParams,
    Part,
    Role,
    SendMessageRequest,
    TextPart,
)
from a2a.utils import new_agent_text_message

# A local port for our demo agent. In a real deployment this would be a
# publicly routable HTTPS URL instead of localhost.
HOST = "127.0.0.1"
PORT = 9999
BASE_URL = f"http://{HOST}:{PORT}"

## Step 1 -- Define the Agent's Logic (`AgentExecutor`)

`AgentExecutor` is the one abstract class every A2A server must implement. A2A itself is agnostic to what happens inside `execute()` -- the protocol only cares that a `Message` or `Task` event eventually lands on the `event_queue` it hands us. That is the seam where "protocol" ends and "agent intelligence" begins.

For this foundations notebook, `execute()` just echoes the user's text back with a prefix -- no LLM call, no tools. `context.get_user_input()` pulls the text out of the incoming `Message`'s `Part`s for us.

In [ ]:
# ============ AGENT EXECUTOR (deliberately trivial "agent logic") ============
class EchoAgentExecutor(AgentExecutor):
    """
    Minimal AgentExecutor used to demonstrate A2A protocol mechanics.

    In production this is where you would plug in a real model, e.g.:

        from helpers import get_llm
        llm = get_llm()
        reply_text = llm.invoke(user_text).content

    Here we skip the LLM entirely so the notebook stays focused on the
    protocol -- discovery, task submission, and the response envelope --
    rather than on agent reasoning.
    """

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_text = context.get_user_input()
        reply_text = f"Echo from A2A agent: {user_text!r}"

        # Publish a plain agent Message as the result of this task turn.
        # (A more elaborate agent could instead publish a Task with
        # TaskStatusUpdateEvent/TaskArtifactUpdateEvent events for
        # long-running or streaming work.)
        await event_queue.enqueue_event(
            new_agent_text_message(
                reply_text,
                context_id=context.context_id,
                task_id=context.task_id,
            )
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        # Cancellation is part of the Task lifecycle (see TaskState.canceled),
        # but this trivial synchronous demo agent has nothing to cancel.
        raise NotImplementedError("EchoAgentExecutor does not support cancellation.")

## Step 2 -- Describe the Agent With an Agent Card

The `AgentCard` is what gets served at `/.well-known/agent.json`. It advertises:
- **identity**: `name`, `description`, `version`
- **connection info**: `url` (where to send requests) and `preferred_transport`/`protocol_version`
- **capabilities**: e.g. whether the agent supports streaming responses
- **skills**: a list of `AgentSkill` entries, each a named ability with example prompts and MIME types

This is the artifact that makes A2A *discoverable and self-describing* -- a client never needs prior, hardcoded knowledge of what an agent can do; it just reads the card.

In [ ]:
# ============ AGENT CARD ============
echo_skill = AgentSkill(
    id="echo",
    name="Echo Text",
    description="Repeats back whatever text it is sent, prefixed with 'Echo from A2A agent:'.",
    tags=["demo", "protocol-foundations"],
    examples=["hello there", "what is A2A?"],
)

agent_card = AgentCard(
    name="A2A Foundations Echo Agent",
    description="A minimal demo agent used to teach A2A protocol mechanics (Agent Card, Task, Message).",
    url=f"{BASE_URL}/",
    version="1.0.0",
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[echo_skill],
)

agent_card

## Step 3 -- Wire the Agent Card + Executor Into an A2A Server

`DefaultRequestHandler` is the piece of `a2a-sdk` that implements the actual JSON-RPC methods (`message/send`, `tasks/get`, `tasks/cancel`, ...) on top of our `AgentExecutor`, backed by a `TaskStore` (here, `InMemoryTaskStore` -- fine for a demo, a real deployment would persist tasks). `A2AStarletteApplication` then wraps the handler and the `AgentCard` into a Starlette ASGI app that exposes:

- `GET /.well-known/agent.json` -- serves the `AgentCard`
- `POST /` -- the JSON-RPC endpoint the client posts `message/send` requests to

In [ ]:
# ============ SERVER ASSEMBLY ============
request_handler = DefaultRequestHandler(
    agent_executor=EchoAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

server_app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
)

asgi_app = server_app.build()

## Step 4 -- Run the Server In-Process

A Jupyter kernel already runs its own `asyncio` event loop, which lets code cells use top-level `await`. Rather than spinning up a separate OS process or thread, we run `uvicorn.Server.serve()` as a background task **on that same loop**, so the notebook can keep acting as the client immediately afterward. In a real deployment this server would run as its own long-lived process (`uvicorn app:asgi_app`).

In [ ]:
# ============ RUN SERVER IN BACKGROUND ============
uvicorn_config = uvicorn.Config(asgi_app, host=HOST, port=PORT, log_level="warning")
uvicorn_server = uvicorn.Server(uvicorn_config)

server_task = asyncio.create_task(uvicorn_server.serve())

# Give the server a moment to bind the port before we start hitting it.
await asyncio.sleep(1)
print(f"A2A demo server listening at {BASE_URL}")

### Discussion of the Output

At this point we have a real, running A2A server. It knows nothing about any client yet -- it is simply waiting for someone to (1) fetch its Agent Card, then (2) send it a task. That is exactly what the rest of this notebook does, playing the role of the *client*.

## Step 5 -- Discovery: Fetch the Agent Card

This is the first thing any A2A client does before talking to an unfamiliar agent: resolve its `AgentCard` from the well-known path. `A2ACardResolver` handles the `GET /.well-known/agent.json` call and parses the JSON into an `AgentCard` object -- the exact same class the server used to construct it.

In [ ]:
# ============ CLIENT: DISCOVERY ============
async with httpx.AsyncClient() as discovery_httpx_client:
    resolver = A2ACardResolver(httpx_client=discovery_httpx_client, base_url=BASE_URL)
    discovered_card = await resolver.get_agent_card()

print("Discovered Agent Card")
print("----------------------")
print(f"name:            {discovered_card.name}")
print(f"description:     {discovered_card.description}")
print(f"url:             {discovered_card.url}")
print(f"version:         {discovered_card.version}")
print(f"protocol_version:{discovered_card.protocol_version}")
print(f"capabilities:    {discovered_card.capabilities}")
print("skills:")
for skill in discovered_card.skills:
    print(f"  - {skill.id}: {skill.name} -- {skill.description} (tags={skill.tags})")

### Discussion of the Output

We never hardcoded any knowledge of the agent's capabilities into the client -- we only knew its base URL. Everything printed above (name, skills, capabilities) came from the card the server published. This is **capability negotiation**: the client now knows *what* the agent can do and *where* to send requests, purely from a well-known, self-describing document.

## Step 6 -- Task Submission: Send a Message to the Agent

With the Agent Card in hand, the client builds a `Message` (role `user`, one `TextPart`), wraps it in `MessageSendParams`, and wraps *that* in a `SendMessageRequest` -- the JSON-RPC envelope for the `message/send` method. `A2AClient.send_message()` posts this to the agent's `url` and returns the parsed response.

In [ ]:
# ============ CLIENT: TASK SUBMISSION ============
user_message = Message(
    role=Role.user,
    parts=[Part(root=TextPart(text="Hello from the A2A client notebook!"))],
    message_id=str(uuid.uuid4()),
)

send_request = SendMessageRequest(
    id=str(uuid.uuid4()),
    params=MessageSendParams(message=user_message),
)

async with httpx.AsyncClient() as client_httpx_client:
    a2a_client = A2AClient(httpx_client=client_httpx_client, agent_card=discovered_card)
    send_response = await a2a_client.send_message(send_request)

send_response

## Step 7 -- Reading the Response in Protocol Terms

`send_response.root` is either a `SendMessageSuccessResponse` (holding `.result`, which can be a `Message` **or** a full `Task`) or a JSON-RPC error response. Our `EchoAgentExecutor` published a plain `Message`, so `.result` here is a `Message` -- if the agent instead needed multiple turns or long-running work, it would return a `Task` object whose `.status.state` walks through the A2A `TaskState` lifecycle (`submitted -> working -> completed`, or `input-required` / `failed` / `canceled`).

In [ ]:
# ============ INSPECT THE RESPONSE ============
result = send_response.root.result

if hasattr(result, "parts"):  # a Message
    reply_text = "".join(
        part.root.text for part in result.parts if hasattr(part.root, "text")
    )
    print(f"Agent replied with a Message (role={result.role}):")
    print(reply_text)
else:  # a Task
    print(f"Agent replied with a Task (id={result.id}, state={result.status.state}):")
    print(result)

### Discussion of the Output

End to end, the client just performed the full A2A protocol handshake with an agent it had no prior integration with:

1. **Discovery** -- `GET /.well-known/agent.json` -> `AgentCard`
2. **Capability negotiation** -- read `capabilities`/`skills` off that card to decide how to talk to the agent
3. **Task submission** -- `POST /` with a `message/send` JSON-RPC request carrying a `Message`
4. **Response** -- a `Message` (or `Task`) came back, containing the agent's reply

Every one of these steps is transport- and vendor-agnostic: the same client code would work against any compliant A2A agent, LangGraph-based or not, written by us or by a third party -- which is the entire point of the protocol.

## Shutting Down the Demo Server

In [ ]:
# ============ CLEANUP ============
uvicorn_server.should_exit = True
await server_task
print("A2A demo server stopped.")

## Summary & Key Takeaways

- **Agent Card** = a self-describing JSON manifest served at `/.well-known/agent.json`; it is how a client discovers *what* an agent can do and *where* to send requests, with zero prior hardcoded integration.
- **Task** = the unit of work sent to an agent, tracked with an `id`/`context_id` and a lifecycle (`submitted -> working -> completed/failed/canceled`, with an `input-required` branch for multi-turn work). Simple exchanges can resolve as a plain `Message` instead of a full `Task`.
- **Message** = the conversational turn itself: a `role` (`user`/`agent`) plus a list of `Part`s (text, file, or data).
- The protocol flow is always **discovery -> capability negotiation -> task submission -> response**, and it is entirely transport/vendor-agnostic -- this is what makes A2A useful for cross-framework agent interop.
- We used the real `a2a-sdk` package throughout: `AgentCard`/`AgentSkill`/`AgentCapabilities`, `AgentExecutor` + `DefaultRequestHandler` + `A2AStarletteApplication` on the server side, and `A2ACardResolver` + `A2AClient` on the client side -- the same package (`a2a-sdk>=0.3.0`) already used in `09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/`.
- The agent's actual "thinking" (`EchoAgentExecutor.execute`) was kept trivial on purpose. A production agent would swap the canned string for a real model call, e.g. `from helpers import get_llm; get_llm().invoke(user_text)`, per this repo's LLM-factory convention.

This is the **foundations** notebook of a 3-part A2A track (`09_Agent_Protocols/A2A/`). Sibling notebooks -- `02_Building_Agents_with_A2A/` (a real LLM agent exposed over A2A) and `03_Applications/` (a cross-framework interop demo) -- are being authored separately and are not created by this notebook.